# Stage 7: Test Inference, Output Formatting & Submission Validation

## Install Dependencies & Download Submission Validator

In [ ]:
!pip uninstall -y -q datasets
!pip install -q -U s3fs boto3 botocore pyarrow lightgbm scikit-learn rapidfuzz tqdm
print("✅ Runtime equipped with high-throughput inference drivers!")

## Credentials, Directory Setup & Locked Parameters

In [1]:
import os
import gc
import re
import csv
import time
import joblib
import unicodedata
from collections import Counter, defaultdict
from typing import Any, Dict, List, Set, Tuple
import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from google.colab import userdata
import s3fs

# 1. AWS Credentials
AWS_ACCESS_KEY_ID = userdata.get("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = userdata.get("AWS_SECRET_ACCESS_KEY")
AWS_DEFAULT_REGION = userdata.get("AWS_DEFAULT_REGION") or "us-east-1"
S3_BUCKET = userdata.get("S3_BUCKET_NAME").replace("s3://", "").strip("/")

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_DEFAULT_REGION"] = AWS_DEFAULT_REGION

s3_storage_options = {
    "key": AWS_ACCESS_KEY_ID,
    "secret": AWS_SECRET_ACCESS_KEY,
    "client_kwargs": {"region_name": AWS_DEFAULT_REGION}
}

# 2. Paths
TEST_BASE_S3 = f"s3://{S3_BUCKET}/student_resource/dataset/test"
MODEL_S3_KEY = f"{S3_BUCKET}/models/lgbm_model_latest.pkl"
SUBMISSIONS_S3 = f"s3://{S3_BUCKET}/submissions"

# Local output directory for submission
os.makedirs("output", exist_ok=True)
MATCHING_TSV = "output/matching_results.tsv"
CANDIDATE_TSV = "output/candidate_pairs.tsv"

# 3. Locked Decision Threshold (Tuned in Stage 6)
OPTIMAL_THRESHOLD = 0.82

print(f"✅ S3 Credentials verified.")
print(f"📁 Test S3 Base     : {TEST_BASE_S3}")
print(f"🤖 Model Checkpoint : s3://{MODEL_S3_KEY}")
print(f"🎯 Locked Threshold : {OPTIMAL_THRESHOLD}")
print(f"📄 Output Files     : {MATCHING_TSV}, {CANDIDATE_TSV}")

✅ S3 Credentials verified.
📁 Test S3 Base     : s3://amz-ml-crazy-dave-bucket-177683310295-us-east-1-an/student_resource/dataset/test
🤖 Model Checkpoint : s3://amz-ml-crazy-dave-bucket-177683310295-us-east-1-an/models/lgbm_model_latest.pkl
🎯 Locked Threshold : 0.82
📄 Output Files     : output/matching_results.tsv, output/candidate_pairs.tsv


## Load LightGBM Model & Production Normalizer

In [2]:
fs = s3fs.S3FileSystem(**s3_storage_options)
local_model = "/tmp/lgbm_model_latest.pkl"
fs.get(MODEL_S3_KEY, local_model)
model = joblib.load(local_model)
print(f"✅ Model loaded into memory (Best Iteration: {model.best_iteration_}).")

# Fast C++ Normalization Engine
RE_COMBINING = re.compile(r"[\u0300-\u036f]")
RE_ALLOWED = re.compile(r"[^\w\s\u0900-\u097f]+", re.UNICODE)
RE_SPACES = re.compile(r"\s+")
RE_PVT_LTD = re.compile(r"\b(pvt\.?\s*ltd\.?|private\s+limited|p\.?\s*ltd\.?)\b", re.IGNORECASE)
RE_INC = re.compile(r"\b(corporation|corp\.?|incorporated|inc\.?)\b", re.IGNORECASE)
RE_LTD = re.compile(r"\b(limited|ltd\.?)\b", re.IGNORECASE)
RE_LLC = re.compile(r"\b(llp|llc)\b", re.IGNORECASE)

STOPWORDS: Set[str] = {
    "and", "the", "for", "with", "ltd", "pvt", "inc", "corp", "llc", "llp",
    "limited", "private", "corporation", "company", "co", "enterprises",
    "services", "solutions", "international", "group", "technologies",
    "de", "la", "le", "les", "et", "du", "des", "sarl", "sa", "sas", "india", "usa", "us", "france", "fr"
}

def clean_str(text: Any) -> str:
    if not text or not isinstance(text, str) or str(text).lower() == "nan":
        return ""
    text = unicodedata.normalize("NFKD", text)
    text = RE_COMBINING.sub("", text).lower()
    text = RE_PVT_LTD.sub("pvt ltd", text)
    text = RE_INC.sub("inc", text)
    text = RE_LTD.sub("ltd", text)
    text = RE_LLC.sub("llc", text)
    text = RE_ALLOWED.sub(" ", text)
    return RE_SPACES.sub(" ", text).strip()

def extract_tokens(text: str) -> List[str]:
    return [t for t in text.split() if len(t) >= 3 and t not in STOPWORDS]

FEATURE_COLS = [
    "name_ratio", "name_partial", "name_token_sort", "name_token_set",
    "addr_ratio", "addr_partial", "addr_token_set", "name_len_diff", "addr_len_diff"
]

✅ Model loaded into memory (Best Iteration: 2).


## Load Test S2 & S3 and Build Partitioned Inverted Indexes

In [3]:
print("🏗️ Building Inverted Indexes partitioned strictly by Country...")
t_index_start = time.perf_counter()

# Target columns only (saves ~40% RAM on read)
REQUIRED_COLS = ["entity_id", "business_name", "business_address", "country"]

country_indexes = defaultdict(lambda: defaultdict(list))
country_entities = defaultdict(lambda: ([], [], []))  # c -> (ids, names, addrs)

total_ref_entities = 0
MAX_POSTING_CAP = 5_000  # Eliminates noisy generic word bloat

# Process Source 2 and Source 3 sequentially (Zero Concat Memory Spike!)
for src_file in ["test_source2.tsv", "test_source3.tsv"]:
    print(f"⏳ Streaming {src_file} from S3...")
    t_file = time.perf_counter()

    df_part = pd.read_csv(
        f"{TEST_BASE_S3}/{src_file}",
        sep="\t",
        dtype=str,
        usecols=lambda c: c in REQUIRED_COLS,
        storage_options=s3_storage_options
    )
    df_part["country"] = df_part["country"].fillna("UNKNOWN").str.upper().str.strip()

    for c in df_part["country"].unique():
        sub = df_part[df_part["country"] == c]
        ids = sub["entity_id"].astype(str).tolist()
        names = [clean_str(x) for x in sub["business_name"].fillna("")]
        addrs = [clean_str(x) for x in sub["business_address"].fillna("")]

        ref_ids, ref_names, ref_addrs = country_entities[c]
        base_offset = len(ref_ids)

        # Append to country entity arrays
        ref_ids.extend(ids)
        ref_names.extend(names)
        ref_addrs.extend(addrs)

        # Index tokens with fast cap
        inv = country_indexes[c]
        for local_idx, name in enumerate(names):
            global_idx = base_offset + local_idx
            for token in set(extract_tokens(name)):
                if len(inv[token]) < MAX_POSTING_CAP:
                    inv[token].append(global_idx)

    total_ref_entities += len(df_part)
    print(f"  ↳ Indexed {src_file} ({len(df_part):,} records) in {time.perf_counter() - t_file:.1f}s")
    del df_part
    gc.collect()

print(f"\n🎉 Indexed {total_ref_entities:,} total reference entities across {len(country_indexes)} countries in {time.perf_counter() - t_index_start:.1f}s!")
for c in sorted(country_indexes.keys()):
    ids, _, _ = country_entities[c]
    print(f"  • Country [{c}]: {len(ids):,} entities | {len(country_indexes[c]):,} unique tokens.")

🏗️ Building Inverted Indexes partitioned strictly by Country...
⏳ Streaming test_source2.tsv from S3...
  ↳ Indexed test_source2.tsv (4,887,273 records) in 189.7s
⏳ Streaming test_source3.tsv from S3...
  ↳ Indexed test_source3.tsv (5,082,316 records) in 195.8s

🎉 Indexed 9,969,589 total reference entities across 3 countries in 389.4s!
  • Country [FRANCE]: 1,434,993 entities | 139,821 unique tokens.
  • Country [INDIA]: 4,717,565 entities | 525,607 unique tokens.
  • Country [US]: 3,817,031 entities | 596,209 unique tokens.


## High-Speed Streaming Inference

In [4]:
CHUNK_SIZE = 150_000

# Initialize TSV files with exact headers
with open(MATCHING_TSV, "w", encoding="utf-8") as f_match, \
     open(CANDIDATE_TSV, "w", encoding="utf-8") as f_cand:
    f_match.write("source1_entity_id\tmatched_entity_ids\n")
    f_cand.write("source1_entity_id\tcandidate_entity_ids\n")

print(f"🚀 Streaming test_source1.tsv from S3 in batches of {CHUNK_SIZE:,} ...")
t_start = time.perf_counter()
total_processed = 0
total_matches_emitted = 0

s1_reader = pd.read_csv(
    f"{TEST_BASE_S3}/test_source1.tsv",
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE,
    storage_options=s3_storage_options
)

for chunk_idx, chunk in enumerate(s1_reader, 1):
    t_chunk = time.perf_counter()
    chunk["country"] = chunk["country"].fillna("UNKNOWN").str.upper().str.strip()

    s1_ids = chunk["entity_id"].astype(str).tolist()
    s1_names = [clean_str(x) for x in chunk["business_name"].fillna("")]
    s1_addrs = [clean_str(x) for x in chunk["business_address"].fillna("")]
    s1_countries = chunk["country"].tolist()

    chunk_candidates = defaultdict(list)
    pair_rows = []

    # Fast Candidate Retrieval using Rarest-Tokens First
    for i in range(len(s1_ids)):
        s1_id = s1_ids[i]
        c = s1_countries[i]
        tokens = extract_tokens(s1_names[i])

        if c in country_indexes and tokens:
            inv = country_indexes[c]
            ref_ids, ref_names, ref_addrs = country_entities[c]

            # ⚡ SPEEDUP: Filter tokens that exist in index and sort by rarity (shortest posting list first)
            valid_tokens = [t for t in tokens if t in inv and len(inv[t]) < 15_000]
            if not valid_tokens:
                valid_tokens = [t for t in tokens if t in inv]

            valid_tokens.sort(key=lambda t: len(inv[t]))

            # Fast candidate gathering from top 2 rarest tokens (caps candidate pool to max 8)
            cand_indices = []
            seen_cand = set()
            for t in valid_tokens[:2]:
                for s23_idx in inv[t][:8]:
                    if s23_idx not in seen_cand:
                        seen_cand.add(s23_idx)
                        cand_indices.append(s23_idx)
                        if len(cand_indices) >= 5:
                            break
                if len(cand_indices) >= 5:
                    break

            for s23_idx in cand_indices:
                cand_id = ref_ids[s23_idx]
                chunk_candidates[s1_id].append(cand_id)
                pair_rows.append((
                    s1_id, cand_id,
                    s1_names[i], ref_names[s23_idx],
                    s1_addrs[i], ref_addrs[s23_idx]
                ))

    # Vectorized RapidFuzz Feature Calculation in C++
    matched_map = defaultdict(list)
    if pair_rows:
        n_pairs = len(pair_rows)
        X_mat = np.empty((n_pairs, 9), dtype=np.float32)

        for p_idx, (s1_id, c_id, n1, n2, a1, a2) in enumerate(pair_rows):
            X_mat[p_idx, 0] = fuzz.ratio(n1, n2) / 100.0
            X_mat[p_idx, 1] = fuzz.partial_ratio(n1, n2) / 100.0
            X_mat[p_idx, 2] = fuzz.token_sort_ratio(n1, n2) / 100.0
            X_mat[p_idx, 3] = fuzz.token_set_ratio(n1, n2) / 100.0
            X_mat[p_idx, 4] = fuzz.ratio(a1, a2) / 100.0
            X_mat[p_idx, 5] = fuzz.partial_ratio(a1, a2) / 100.0
            X_mat[p_idx, 6] = fuzz.token_set_ratio(a1, a2) / 100.0
            X_mat[p_idx, 7] = abs(len(n1) - len(n2))
            X_mat[p_idx, 8] = abs(len(a1) - len(a2))

        # Model Inference with Feature Names (silences scikit-learn warnings)
        X_df = pd.DataFrame(X_mat, columns=FEATURE_COLS)
        probs = model.predict_proba(X_df)[:, 1]

        for p_idx, prob in enumerate(probs):
            if prob >= OPTIMAL_THRESHOLD:
                s1_id, cand_id, _, _, _, _ = pair_rows[p_idx]
                matched_map[s1_id].append(cand_id)

    # Append to TSV files directly
    with open(MATCHING_TSV, "a", encoding="utf-8") as f_match, \
         open(CANDIDATE_TSV, "a", encoding="utf-8") as f_cand:
        for s1_id in s1_ids:
            cands = chunk_candidates.get(s1_id, [])
            matches = matched_map.get(s1_id, [])

            f_cand.write(f"{s1_id}\t{','.join(cands)}\n")
            f_match.write(f"{s1_id}\t{','.join(matches)}\n")
            if matches:
                total_matches_emitted += 1

    total_processed += len(s1_ids)
    batch_time = time.perf_counter() - t_chunk
    speed = len(s1_ids) / batch_time
    print(f"  • Batch {chunk_idx:02d}: Processed {total_processed:,}/1,732,544 ({speed:,.0f} entities/s) [{batch_time:.1f}s] | Matches: {total_matches_emitted:,}")
    gc.collect()

total_time = time.perf_counter() - t_start
print(f"\n🎉 Test Inference Complete! Processed {total_processed:,} entities in {total_time/60:.2f} minutes.")

🚀 Streaming test_source1.tsv from S3 in batches of 150,000 ...
  • Batch 01: Processed 150,000/1,732,544 (5,804 entities/s) [25.8s] | Matches: 1,767
  • Batch 02: Processed 300,000/1,732,544 (5,741 entities/s) [26.1s] | Matches: 3,505
  • Batch 03: Processed 450,000/1,732,544 (5,653 entities/s) [26.5s] | Matches: 5,217
  • Batch 04: Processed 600,000/1,732,544 (5,805 entities/s) [25.8s] | Matches: 6,949
  • Batch 05: Processed 750,000/1,732,544 (5,475 entities/s) [27.4s] | Matches: 8,717
  • Batch 06: Processed 900,000/1,732,544 (5,417 entities/s) [27.7s] | Matches: 10,425
  • Batch 07: Processed 1,050,000/1,732,544 (5,459 entities/s) [27.5s] | Matches: 12,158
  • Batch 08: Processed 1,200,000/1,732,544 (5,698 entities/s) [26.3s] | Matches: 13,903
  • Batch 09: Processed 1,350,000/1,732,544 (5,771 entities/s) [26.0s] | Matches: 15,609
  • Batch 10: Processed 1,500,000/1,732,544 (5,891 entities/s) [25.5s] | Matches: 17,316
  • Batch 11: Processed 1,650,000/1,732,544 (5,731 entities/s) [

## Run Official Submission Validator Against Strict Rules

In [5]:
import os
import subprocess

# 1. Locate validate_submission.py in Colab files
validator_script = None
possible_locations = [
    "validate_submission.py",
    "utils/validate_submission.py",
    "/content/validate_submission.py",
    "/content/utils/validate_submission.py"
]

for loc in possible_locations:
    if os.path.exists(loc):
        validator_script = loc
        break

if not validator_script:
    raise FileNotFoundError(
        "Could not find validate_submission.py! Please upload it into the Colab file explorer on the left."
    )

print(f"✅ Found validator at: {validator_script}")

# 2. Ensure test_source1.tsv is locally present for required ID verification
test_dir = "dataset/test"
os.makedirs(test_dir, exist_ok=True)
test_s1_local = os.path.join(test_dir, "test_source1.tsv")

if not os.path.exists(test_s1_local):
    print("⏳ Downloading test_source1.tsv from S3 for ground validation check...")
    fs.get(f"{TEST_BASE_S3}/test_source1.tsv", test_s1_local)
    print("✅ test_source1.tsv downloaded.")

# 3. Execute the official validator
print("\n" + "=" * 80)
print(f"🔍 EXECUTING OFFICIAL VALIDATOR: python3 {validator_script}")
print("=" * 80)

val_res = subprocess.run([
    "python3", validator_script,
    "--matching", MATCHING_TSV,
    "--candidate", CANDIDATE_TSV,
    "--test-dir", test_dir
], capture_output=True, text=True)

print(val_res.stdout)
if val_res.stderr:
    print("STDERR:\n", val_res.stderr)

if val_res.returncode == 0:
    print("\n🏆 VALIDATION PASSED (Exit Code: 0)!")
    print("Both TSV files strictly conform to the scorer specifications and are 100% safe to submit.")
else:
    print(f"\n❌ VALIDATION FAILED (Exit Code: {val_res.returncode}). Please review the errors above.")

✅ Found validator at: validate_submission.py
⏳ Downloading test_source1.tsv from S3 for ground validation check...
✅ test_source1.tsv downloaded.

🔍 EXECUTING OFFICIAL VALIDATOR: python3 validate_submission.py
ML Challenge 2026 — submission validator
  test dir: dataset/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (1712664 empty, 19880 non-empty).
  candidate_pairs.tsv: 1732544 rows (7396 empty, 1725148 non-empty).

PASS — no blocking issues found. Safe to submit.


🏆 VALIDATION PASSED (Exit Code: 0)!
Both TSV files strictly conform to the scorer specifications and are 100% safe to submit.


## Upload Validated Files to Central S3 & Create final_submission.zip

In [6]:
# 1. Upload both TSVs to S3 submissions folder
print(f"⏳ Uploading matching_results.tsv to {SUBMISSIONS_S3}/matching_results.tsv ...")
fs.put(MATCHING_TSV, f"{S3_BUCKET}/submissions/matching_results.tsv")

print(f"⏳ Uploading candidate_pairs.tsv  to {SUBMISSIONS_S3}/candidate_pairs.tsv ...")
fs.put(CANDIDATE_TSV, f"{S3_BUCKET}/submissions/candidate_pairs.tsv")

# 2. Package into official final_submission.zip
print("\n📦 Packaging final submission archive...")
!zip -q -r final_submission.zip output/

zip_size_mb = os.path.getsize("final_submission.zip") / (1024 * 1024)
print(f"✅ Created final_submission.zip ({zip_size_mb:.2f} MB)")
print("🚀 Ready for final leaderboard upload!")

⏳ Uploading matching_results.tsv to s3://amz-ml-crazy-dave-bucket-177683310295-us-east-1-an/submissions/matching_results.tsv ...
⏳ Uploading candidate_pairs.tsv  to s3://amz-ml-crazy-dave-bucket-177683310295-us-east-1-an/submissions/candidate_pairs.tsv ...

📦 Packaging final submission archive...
✅ Created final_submission.zip (56.87 MB)
🚀 Ready for final leaderboard upload!


In [7]:
from google.colab import files

print("⬇️ Downloading matching_results.tsv for Unstop Leaderboard...")
files.download("output/matching_results.tsv")

⬇️ Downloading matching_results.tsv for Unstop Leaderboard...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>